# 欠損値の把握
データの中の空欄(欠損値)が、どの列にどれくらいあるかを調べます。欠損の多さだけでなく、「どんな乗客で欠損しているか」にも偏りがあることを確かめ、次の講座での欠損値の扱い方につなげます。

## 学習目標
- 列ごとの欠損値の数と割合を求め、多い順に並べられる
- グループごとの欠損の割合を比べ、欠損の偏りを見つけられる
- 「欠損していること」自体が情報を持つ場合があることを説明できる

## 欠損値を数える

pandas では、空欄は `NaN`(数値の列)などの欠損値として読み込まれます。`isna()` は各セルが欠損かどうか(True / False)を返し、`sum()` で列ごとの True の数、つまり欠損値の数が求まります。

In [1]:
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# isna() で各セルが欠損なら True にし、sum() で列ごとに True の数(=欠損値の数)を数える
train.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            173
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          695
Embarked         2
Boat           566
Body           811
HomeDest       392
dtype: int64


数より割合の方が比べやすいので、`mean()` で欠損の割合を求め、多い順に並べます(True は 1、False は 0 として平均されます)。

In [2]:
# mean() で列ごとの欠損の割合を求める(True は 1、False は 0 として平均される)
missing_rate = train.isna().mean()
# sort_values(ascending=False) で大きい順に並べ、round(3) で小数第3位までにする
missing_rate.sort_values(ascending=False).round(3)

Body           0.910
Cabin          0.780
Boat           0.635
HomeDest       0.440
Age            0.194
Embarked       0.002
Fare           0.001
PassengerId    0.000
Survived       0.000
Pclass         0.000
Name           0.000
Sex            0.000
SibSp          0.000
Parch          0.000
Ticket         0.000
dtype: float64


| 列 | 欠損の割合 | 読み取り |
|---|---|---|
| `Body` | 91% | 遺体が見つかった人にだけある番号。予測には使わない(リーク) |
| `Cabin` | 78% | 大部分が欠損。そのままでは使いにくい |
| `Boat` | 64% | 救命ボートに乗らなかった人は空欄。予測には使わない(リーク) |
| `Age` | 19% | 約 5 人に 1 人の年齢が不明。補完の方法を考える必要がある |
| `Embarked`・`Fare` | 0.2% 以下 | ごくわずか |

## 欠損の偏りを調べる

欠損は、すべての乗客で同じように起きているとは限りません。客室の等級ごとに `Age` の欠損の割合を比べてみます。

In [3]:
# groupby("Pclass") で等級ごとのグループに分け、各グループの Age 列を取り出す
# apply には「グループの Age 列 s を受け取って、欠損の割合を返す関数」を渡す
# (lambda s: ... は、その場で作る小さな関数の書き方)
train.groupby("Pclass")["Age"].apply(lambda s: s.isna().mean()).round(3)

Pclass
1    0.097
2    0.051
3    0.297
Name: Age, dtype: float64


3 等の乗客は約 30% の年齢が不明で、1 等・2 等(5〜10%)よりずっと多くなっています。このように欠損に偏りがあると、「欠損した行を削除する」だけでは 3 等の乗客が減り、データの傾向がゆがんでしまいます。

## 欠損そのものが持つ情報

`Cabin` が記録されているかどうかで、生存率を比べてみます。`notna()` は欠損でないときに True を返します。

In [4]:
# 客室番号が記録されているか(True / False)でグループに分ける
has_cabin = train["Cabin"].notna()
# Survived は 0 / 1 なので、グループごとの平均が生存率になる
train.groupby(has_cabin)["Survived"].mean().round(3)

Cabin
False    0.299
True     0.638
Name: Survived, dtype: float64


客室番号が記録されている乗客の生存率は 64% で、記録されていない乗客(30%)の約 2 倍です。客室番号が記録されているのは主に 1 等の乗客であるためと考えられます。このように、**「欠損しているかどうか」自体が特徴になる** ことがあります(次の講座で特徴量として使います)。

## 演習

予測用のデータ `test.csv` について、列ごとの欠損値の **割合** を求め、多い順に並べて、小数第3位まで表示してください。

In [ ]:
# @starter
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# 列ごとの欠損の割合を求める
# (1つ目の ____ に「欠損かどうかを判定する」メソッド、2つ目の ____ に「平均を求める」メソッドを入れる)
missing = test.____().____()

# 大きい順に並べ、小数第3位まで表示する
print(missing.sort_values(ascending=False).round(3))

In [5]:
# @solution
# @title: 予測用データの欠損の割合
# @hint: isna() で欠損かどうかを判定し、mean() で列ごとの割合にします。
# @explanation: 予測用のデータでは Cabin が約 76%、Age が約 22% 欠損しています。学習用のデータと同じ傾向です。予測用のデータにも欠損があるので、モデルに入れる前に同じ方法で補完する必要があります。
import pandas as pd

# 予測用のデータを読み込む
test = pd.read_csv("test.csv")

# isna() で欠損かどうかを判定し、mean() で列ごとの欠損の割合を求める
missing = test.isna().mean()

# 大きい順に並べ、小数第3位まで表示する
print(missing.sort_values(ascending=False).round(3))

Cabin          0.763
Age            0.215
PassengerId    0.000
Pclass         0.000
Name           0.000
Sex            0.000
SibSp          0.000
Parch          0.000
Ticket         0.000
Fare           0.000
Embarked       0.000
dtype: float64


学習用のデータ `train.csv` について、**性別(`Sex`)ごと** の `Age` の欠損の割合を求め、小数第3位まで表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# グループごとに Age の欠損の割合を求める(____ に、グループ分けに使う列の名前を入れる)
rate = train.groupby("____")["Age"].apply(lambda s: s.isna().mean())

# 小数第3位まで表示する
print(rate.round(3))

In [6]:
# @solution
# @title: 性別ごとの年齢の欠損の割合
# @hint: groupby の引数にグループ分けに使う列名を指定します。
# @explanation: 年齢の欠損の割合は、女性が約 15%、男性が約 22% で、等級ほどの大きな差はありません。
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 性別ごとのグループに分け、Age の欠損の割合を求める
rate = train.groupby("Sex")["Age"].apply(lambda s: s.isna().mean())

# 小数第3位まで表示する
print(rate.round(3))

Sex
female    0.148
male      0.218
Name: Age, dtype: float64


<!-- @question
type: choice
answer: 3 等の乗客の割合が減り、生存率などの傾向がゆがむおそれがある。
title: 欠損した行を削除する問題点
hint: 年齢の欠損は、どの等級の乗客に多かったかを思い出しましょう。
explanation: 年齢の欠損は 3 等の乗客に多い(約 30%)ため、年齢が欠損した行を削除すると 3 等の乗客が特に多く減ります。3 等の乗客は生存率が低いので、残ったデータの生存率は実際より高くなるなど、傾向がゆがみます。また、予測用のデータにも年齢の欠損があるため、削除では予測ができない乗客が出てしまいます。
-->
学習用のデータで、`Age` が欠損している行をすべて削除してから分析することにした。この方法の問題点として最も適切なものを選べ。

- 年齢の欠損はどの等級でも同じ割合なので、問題はない。
- 3 等の乗客の割合が減り、生存率などの傾向がゆがむおそれがある。
- 削除すると列の数が減るので、モデルが作れなくなる。
- 欠損値は自動的に 0 として計算されるので、削除する必要はない。

## まとめ

- `isna().sum()` で欠損値の数、`isna().mean()` で割合を求める。
- 欠損はグループによって偏ることがある(年齢は 3 等の乗客で多く欠損)。欠損した行の削除は傾向をゆがめるおそれがある。
- 「欠損しているかどうか」自体が、生存と関係する情報になることがある(客室番号の有無)。